<a href="https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

## My provisional lane: Lane 2 — Refresh / Content Opportunity Scoring

I want to investigate which content pages should be reviewed first for possible refresh or other content action. I chose this lane because the starter dataset contains page-level search and content performance signals that can help prioritize limited editorial time. Instead of treating every declining or low-performing page as equally important, the goal is to produce a ranked queue that helps identify the pages most worth reviewing first.

The starter pipeline also makes this lane worth investigating. In the starter experiment, the hand-written baseline achieved Precision@50 of 0.240, while the Random Forest achieved Precision@50 of 0.740. This observed result suggests that combining multiple signals may be more useful for prioritization than relying on one fixed hand-written rule. My lane is provisional and may be refined as I learn more about the warehouse data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
pd.options.display.max_columns = 200
df = pd.read_csv("content_refresh_anonymized.csv")
df.shape, df.head(3)

((30000, 44),
              content_id          client_id  search_volume  competition  \
 0  content_304f48230142  client_f369cb89fc           10.0         0.67   
 1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
 2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
 
   competition_level   cpc     content_type    main_intent  word_count  \
 0              HIGH  2.05  keyword article  transactional      3221.0   
 1               LOW  0.05  keyword article  informational      2481.0   
 2               LOW  0.00  keyword article  informational      3515.0   
 
    char_count provider_used              model_used  impressions_90d  \
 0     20457.0           NaN        gemini-2.5-flash             3803   
 1     15562.0           NaN  gemini-3-flash-preview            15320   
 2     23643.0           NaN        gemini-2.5-flash            12581   
 
    clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  \
 0          

## 2. The question: decision, action, cost of a wrong call

## Research question

Which content pages should an editor review first for refresh or other content action, based on safe search and content performance signals?

**Unit of analysis:** One pseudonymized content item/page.

**Decision:** Given limited editorial resources, which pages should be prioritized for review first?

**Who acts:** A content editor or SEO/content team.

**Action:** Review the highest-priority pages and decide whether they should be refreshed, expanded, protected, pruned, or monitored.

**Output:** A ranked page-level opportunity queue with a priority score, suggested review action, and human-readable reason codes.

**Cost of a wrong call:** A false positive could waste limited editor time reviewing a page that is not actually a valuable opportunity. A false negative could cause the team to miss a genuinely important page that should have been reviewed. The final evaluation should therefore focus on whether the top-ranked recommendations are genuinely useful for prioritization.

**Why data or ML may help:** A simple rule can use one or two signals, but page prioritization may depend on multiple signals interacting together. A learned model may identify useful patterns that are difficult to capture with one fixed rule. It will be compared with a transparent baseline rather than assuming ML is automatically better.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
n_rows = len(df)
n_clients = df['client_id'].nunique()
with_impr_frac = (df['impressions_90d'] > 0).mean()
print(f"rows: {n_rows:,}, unique clients: {n_clients:,}")
print(f"fraction with impressions_90d > 0: {with_impr_frac:.3f} ({with_impr_frac*100:.1f}%)")

rows: 30,000, unique clients: 32
fraction with impressions_90d > 0: 1.000 (100.0%)


## 3. Quick look at the data (2-3 real numbers)

I loaded the starter CSV and produced a few summary numbers below that show this lane is viable

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
n_500 = (df['impressions_90d'] >= 500).sum()
n_100 = (df['impressions_90d'] >= 100).sum()
print(f"rows with impressions_90d >= 500: {n_500:,}")
print(f"rows with impressions_90d >= 100: {n_100:,}")

rows with impressions_90d >= 500: 16,726
rows with impressions_90d >= 100: 22,006


## 4. Careful words: what I can and can't claim

What I can claim: Observational associations and a ranked decision-support queue that prioritizes pages for manual review. I can show which signals correlate with the starter label and produce a transparent baseline and a model that improves precision@K on the starter slice.
        What I can't claim: Causal claims that a refresh will cause recovery (no experiment). I can't reconstruct raw URLs, client names, or publish private data. The starter label is a current-window proxy (trend_direction); it's not a future-window ground truth. There are gotchas: rate columns are percent-like (ctr=0.76 means 0.76%), avg_position=0 means no data, and trend_pct/trend_direction relationships can leak information if used incorrectly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
if 'is_declining_label' in df.columns:
    print("is_declining_label distribution:")
    print(df['is_declining_label'].value_counts(normalize=True))
else:
    print("trend_direction distribution:")
    print(df['trend_direction'].value_counts(normalize=True))

# CTR gotcha and avg_position no-data
mean_ctr_raw = df['ctr'].dropna().mean()
n_avgpos0 = (df['avg_position'] == 0).sum()
print(f"mean raw ctr value: {mean_ctr_raw:.4f} (interpret as {mean_ctr_raw/100:.4f} fraction)")
print(f"avg_position == 0 rows (no-data): {n_avgpos0:,} ({n_avgpos0/len(df):.3%})")

trend_direction distribution:
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64
mean raw ctr value: 0.5107 (interpret as 0.0051 fraction)
avg_position == 0 rows (no-data): 1,205 (4.017%)


## Self-check

Before you submit, confirm each line honestly:

- [+ ] Every section above is filled — markdown thinking AND the code that backs it
- [+ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+ ] No client names, URLs, or private queries anywhere
- [+ ] My claims use careful words: observed, measured, directional, decision-support
- [ +] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.